In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr

from pyS3M import IOFunctions

IO = IOFunctions.IO_Functions()

from pyS3M import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from pyS3M import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from pyS3M import PlottingBase

plotter = PlottingBase.PublicationPlotter()

from src import AnalysisFunctions

AF = AnalysisFunctions.Analysis_Functions()

import colour

from colour_demosaicing import (
    ROOT_RESOURCES_EXAMPLES,
    demosaicing_CFA_Bayer_bilinear,
    demosaicing_CFA_Bayer_Malvar2004,
    demosaicing_CFA_Bayer_Menon2007,
    mosaicing_CFA_Bayer,
)

from pyS3M import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

In [2]:
data_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

In [3]:
image_size = 40

from Camera_QE import getpixelefficiency

gpe = getpixelefficiency.GPE()
R, G, B, wavelength = gpe.getpixelefficiency("Camera_QE/CS505CU_QE.csv")
RGB = np.vstack([R, G, B]).T
filters = np.ones_like(wavelength)
masks = MSF.MF.get_masks(MSF.mosaic_unit, image_size, image_size)
masks_3d = np.dstack([masks["maskR"], masks["maskG"], masks["maskB"]])
wavelength = wavelength
absolute_QYs = np.vstack([B, G, R])
camera_calibration = {}
camera_calibration["gain"] = gain[:image_size, :image_size]
camera_calibration["offset"] = offset[:image_size, :image_size]
camera_calibration["variance"] = variance[:image_size, :image_size]
camera_calibration["readnoise"] = readnoise[:image_size, :image_size]
camera_calibration["rqe"] = rqe[:image_size, :image_size]
NA = 1.49
pixel_size = 69

In [4]:
n_frames = 200000

In [5]:
params_tosave = np.zeros([7, n_frames])

In [6]:
linker_length_array = np.array([600, 40, 300, 20, 80])
start = time.time()
for ll, linker_length in enumerate(linker_length_array):
    save_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241025/data"
    save_name = (
        "Test_9colourPAINT_linkerlength_pixelsize69nm_" + str(int(linker_length)) + "nm"
    )
    save_overall_name = os.path.join(save_folder, save_name)
    raw_data, photoelectron_data, smoothed_data, weights_map = (
        IO.read_tiff_tophotoelectrons(
            save_overall_name + "_imagestack.tif",
            gain_map=camera_calibration["gain"],
            offset_map=camera_calibration["offset"],
            variance_map=camera_calibration["variance"],
            rqe=camera_calibration["rqe"],
            read_noise=camera_calibration["readnoise"],
        )
    )

    for i in np.arange(n_frames):
        smoothed = sCMOS.var_weighted_uniform_filter(
            photoelectron_data[:, :, i], camera_calibration["variance"], 4
        )
        xc_ig, yc_ig = np.unravel_index(
            np.argmax(smoothed),
            smoothed.shape,
        )
        A = np.sum(np.abs(smoothed))
        sigma = 3
        b = 0
        initial_guess = np.array([xc_ig, yc_ig, A, sigma, b, 0.5, 0.5, 0.5])
        result = AF.WLS_fit(
            photoelectron_data[:, :, i],
            initial_guess,
            masks=masks_3d,
            weights=weights_map[:, :, i],
            display=False,
        )
        params_tosave[:4, i] = result.x[:4]
        params_tosave[4:, i] = result.x[-3:]

        if (i + 1) % 20 == 0:
            print(
                "Analysed frame {}/{} of {}/{} set,  Time elapsed: {:.3f} min".format(
                    i + 1,
                    n_frames,
                    ll + 1,
                    len(linker_length_array),
                    (time.time() - start) / 60.0,
                ),
                end="\r",
                flush=True,
            )

    del photoelectron_data
    del weights_map
    parameters_tosave = np.array(
        ["frame_number", "xc", "yc", "A", "sigma", "R", "G", "B"]
    )
    saving_params = pl.DataFrame(
        data=np.vstack([np.arange(n_frames), params_tosave]).T,
        schema=list(parameters_tosave),
    )

    saving_params.write_csv(os.path.join(save_overall_name + "_data_analysis.csv"))